# EXP-2026-009 — Q5-D negative-control null 아티팩트 복구 (미실행 템플릿)

이 노트북은 **포장 복구**다. 과학 실험이 아니다.

- beat join 재실행 없음 · null 재실행 없음 · `J` 값 계산 없음
- 기존 Drive bundle·shard **수정·삭제·덮어쓰기 없음**
- 새 corrective 폴더 하나만 만들고, 거기에 **`BUNDLE_FILES` 12개 외에는 아무것도 넣지 않는다**

명세: `experiments/specs/EXP-2026-009-q5d-null-artifact-repair.md`
판정 출처: `EXP-2026-008` 실행계약 Decision log의 `P2_PRODUCER_ARTIFACT_OMISSION`

**현재 실행 승인은 없다.** 모듈의 `EXECUTION_APPROVAL_RECORD['granted']` 가
`False` 이므로 아래 실행 셀은 `REPAIR_NOT_APPROVED` 로 거부된다. 승인은 별도
결정이고 별도 PR이 연다 — 이 노트북은 그것을 우회하지 않는다.

## 실패 시 무슨 일이 일어나는가 (읽고 실행하라)

- **target 폴더 생성 전 실패** → 폴더가 아예 만들어지지 않는다.
- **target 폴더 생성 후 실패** → 그 폴더를 **그 자리에 그대로 보존**한다.
  경로와 파일 목록을 보고하고 `REPAIR_INCOMPLETE_TARGET_PRESERVED` 로 표시한다.
  **COMMITTED 도 accepted 도 아니고 등록 대상도 아니다.**
- 이 모듈은 **어떤 경우에도 삭제·덮어쓰기·rename 을 하지 않는다.**
- **재시도는 반드시 새 고유 경로**로 한다. 보존된 폴더를 청소하거나 이어서
  쓰지 않는다.

In [ ]:
# 1. PINNED CHECKOUT — 실행은 움직이는 main 이 아니라 승인 시점의 정확한 커밋에
#    고정한다. 브랜치를 따라가면 승인받지 않은 코드가 실행될 수 있다.
import os, sys, json, subprocess

PINNED_COMMIT = ''     # 승인된 커밋 SHA 를 여기에 적는다 (40-hex). 비우면 거부.
REPO_URL = 'https://github.com/ehdbddl06001-ui/my-github-test.git'
CLONE_TO = '/content/repo'
NEEDED = ('q5d_order_preserving_beat_join.py', 'q5d_null_artifact_repair.py')

if not PINNED_COMMIT or len(PINNED_COMMIT) != 40:
    raise RuntimeError(
        '실행하려면 승인된 정확한 커밋 SHA(40-hex)를 PINNED_COMMIT 에 적어야 '
        '한다. 브랜치 이름은 시간에 따라 움직이므로 고정이 아니다.')


def _is_repo(path):
    if not path:
        return False
    here = os.path.join(path, 'mit-bih')
    return all(os.path.isfile(os.path.join(here, n)) for n in NEEDED)


if not _is_repo(CLONE_TO):
    print('clone:', REPO_URL, '→', CLONE_TO)
    _r = subprocess.run(['git', 'clone', REPO_URL, CLONE_TO],
                        capture_output=True, text=True)
    print((_r.stdout or '').strip() or (_r.stderr or '').strip())

_r = subprocess.run(['git', '-C', CLONE_TO, 'checkout', '--detach',
                     PINNED_COMMIT], capture_output=True, text=True)
print((_r.stdout or '').strip() or (_r.stderr or '').strip())
if _r.returncode != 0:
    raise RuntimeError(f'승인된 커밋 {PINNED_COMMIT} 을 체크아웃하지 못했다.')

_head = subprocess.run(['git', '-C', CLONE_TO, 'rev-parse', 'HEAD'],
                       capture_output=True, text=True).stdout.strip()
if _head != PINNED_COMMIT:
    raise RuntimeError(f'HEAD 가 {_head} 로 승인 커밋과 다르다.')

_dirty = subprocess.run(['git', '-C', CLONE_TO, 'status', '--porcelain'],
                        capture_output=True, text=True).stdout.strip()
if _dirty:
    raise RuntimeError(f'작업 트리가 깨끗하지 않다 — 커밋 고정이 무의미해진다:\n{_dirty}')

REPO = CLONE_TO
if not _is_repo(REPO):
    raise RuntimeError(f'{REPO} 에 필요한 모듈이 없다: {NEEDED}')
_MITBIH = os.path.join(REPO, 'mit-bih')
if _MITBIH not in sys.path:
    sys.path.insert(0, _MITBIH)

import q5d_order_preserving_beat_join as BJ
import q5d_null_artifact_repair as R

MISSING = [n for n in R.module_capabilities() if not hasattr(R, n)]
assert not MISSING, f'stale repair clone, missing {MISSING}'

print('REPO   :', REPO)
print('HEAD   :', _head)
print()
print(R.design_card())

In [ ]:
# 2. 아티팩트 신원 재확인 — 커밋을 아는 것과 디스크의 파일이 그 커밋의 것임을
#    아는 것은 다르다. 모듈·명세·노트북의 SHA 를 체크아웃 뒤에 다시 잰다.
#    등록 identity 는 LF 정규화 SHA 이고, raw byte SHA 도 함께 보고한다.
IDENTITIES = R.artifact_identities(REPO)
for _label in ('module', 'spec', 'notebook'):
    _e = IDENTITIES[_label]
    print(f"{_label:9s} {_e['path']}")
    print(f"          LF  {_e['lf_normalized_sha256']}")
    print(f"          raw {_e['raw_sha256']}  (CRLF: {_e['had_crlf']})")

print()
FROZEN = R.assert_frozen_q5d_unchanged()
print('frozen Q5-D LF  :', FROZEN['lf_normalized_sha256'])
print('frozen Q5-D raw :', FROZEN['raw_sha256'], '(CRLF:', FROZEN['had_crlf'], ')')
print('등록 identity   :', R.FROZEN_Q5D_SHA256_LF, '← LF 정규화 기준')
print('rule fingerprint:', FROZEN['rule_fingerprint'])
print()
print(R.NEWLINE_CONVENTION)

In [ ]:
# 3. 합성 fixture 검증 — 실제 자산을 열기 전에 통과해야 하는 가장 싼 관문.
#    실패하면 stderr 와 종료 코드를 보이고 여기서 멈춘다(조용한 실패 금지).
_res = subprocess.run(
    [sys.executable, os.path.join(REPO, 'mit-bih',
                                  'test_q5d_null_artifact_repair.py')],
    capture_output=True, text=True)
print((_res.stdout or '').strip() or '(stdout 없음)')
if _res.returncode != 0:
    print('--- stderr ---')
    print((_res.stderr or '').strip() or '(stderr 없음)')
    raise RuntimeError(
        f'합성 fixture 가 실패했다 (exit {_res.returncode}). '
        f'실행 셀을 누르지 마라 — 실패한 코드로 실제 자산을 열지 않는다.')

In [ ]:
# 4. Drive folder ID 브리지 — 폴더는 **ID 로만** 지정한다. 이름이 같은 다른
#    폴더는 절대 대체물로 받지 않는다. 마운트 경로는 folder ID inventory 와
#    파일 단위(이름·크기·provider checksum)로 연결됐을 때만 인정된다.
#
#    아직 아무것도 열지 않는다. 여기서는 ID 와 경로만 선언한다.
from google.colab import drive as _drive          # noqa: E402
# drive.mount('/content/drive')                   # 승인 후 주석 해제

DRIVE_ROOT = '/content/drive/MyDrive/MedKOS/ecg-model'

SOURCE_FOLDER_ID = R.SOURCE_BUNDLE_FOLDER_ID   # 1JjwBhU8BXf8lRrYPcM2UjFNdIKxE9Ghd
SHARD_FOLDER_ID  = R.SHARD_FOLDER_ID           # 1c0AbOwwu1UoZ_8Wz60fhzjDcgCkLHMG9
RUNS_PARENT_ID   = R.RUNS_PARENT_FOLDER_ID     # 1YbNX4IeWUph3VFwgpCHGFiibzihF6gXh

RUNS_PARENT_DIR = f'{DRIVE_ROOT}/runs'
SOURCE_DIR = ''   # 등록 folder ID 에 대응하는 마운트 경로 (읽기 전용)
SHARD_DIR  = ''   # 등록 shard folder ID 에 대응하는 마운트 경로 (읽기 전용)
TARGET_DIR = ''   # 새 corrective 폴더 — RUNS_PARENT_DIR 바로 아래, 아직 없는 이름

for _label, _value in (('SOURCE_FOLDER_ID', SOURCE_FOLDER_ID),
                       ('SHARD_FOLDER_ID', SHARD_FOLDER_ID),
                       ('RUNS_PARENT_ID', RUNS_PARENT_ID),
                       ('SOURCE_DIR', SOURCE_DIR), ('SHARD_DIR', SHARD_DIR),
                       ('TARGET_DIR', TARGET_DIR)):
    print(f'{_label:17s}:', _value or '(미지정)')

print()
print('shard 계약 :', R.EXPECTED_SHARD_COUNT, '개 —',
      R.EXPECTED_SHARD_FILENAMES[0], '…', R.EXPECTED_SHARD_FILENAMES[-1])
print('bundle 계약:', len(R.BUNDLE_FILES), '개 · source', len(R.SOURCE_BUNDLE_FILES), '개')
print('NPZ 배열   :', list(R.NPZ_ARRAYS), '· float64', f'({R.N_REPLICATES},)')
print()
print(R.MEMBER_NAMING_NOTE)
print()
print('실행 승인 :', bool(R.EXECUTION_APPROVAL_RECORD.get('granted')))
print(R.APPROVAL_NOTE)

In [ ]:
# 5. 읽기 전용 Drive 서비스 — folder ID inventory 전용. 쓰기 메서드는 쓰지 않는다.
#    승인 전에는 아래 실행 셀이 어차피 거부되므로 어댑터도 소용이 없다.
ADAPTER = None
# 승인 후:
#   from googleapiclient.discovery import build
#   from google.colab import auth
#   auth.authenticate_user()
#   _service = build('drive', 'v3')
#   ADAPTER = R.GoogleDriveFolderInventory(_service)
print('adapter:', ADAPTER)

In [ ]:
# 6. 실행 — 승인 전에는 REPAIR_NOT_APPROVED 로 거부된다. 그것이 정상 동작이다.
#    route: folder ID 브리지 → source snapshot(1회 읽기) → shard 자격검증 →
#           재구성 → null_summary 대조 → NPZ 계약(독립 reader + numpy) →
#           corrective 폴더 조립 → 재검증 → source 재해시 → 새 folder ID 확인.
#    NPZ 계약 통과 전에는 target 폴더를 만들지 않는다.
APPROVAL = R.EXECUTION_APPROVAL_TOKEN   # 리터럴을 적지 않는다

DECISION, FAILURE = None, None
try:
    DECISION = R.run_repair(
        SHARD_DIR, SOURCE_DIR, TARGET_DIR, APPROVAL,
        adapter=ADAPTER, runs_parent_dir=RUNS_PARENT_DIR,
        require_numpy=True, repo_root=REPO)
    print('status:', DECISION['status'])
except R.RepairError as _error:
    FAILURE = _error.as_record()
    print('STOP:', FAILURE['first_stopping_reason'])
    print(FAILURE['message'])
    if FAILURE['incomplete_directory']:
        print()
        print('보존된 미완성 폴더 (삭제하지 않았다):')
        print('  경로 :', FAILURE['incomplete_directory'])
        print('  파일 :', FAILURE['incomplete_listing'])
        print('  상태 :', FAILURE['target_state'],
              '— COMMITTED 아님 · accepted 아님 · 등록 대상 아님')
        print()
        print('재시도는 새 고유 경로로 한다. 이 폴더를 청소하거나 이어 쓰지 마라.')

In [ ]:
# 7. 보고 — 이 셀의 저장된 출력이 외부 기록이다(corrective 폴더 안에는
#    provenance 파일을 넣지 않는다). 여기 나온 folder ID 와 digest 를 별도 PR 이
#    Decision log · ASSETS.md · PROJECT_STATE.md 로 옮긴다.
if DECISION is None:
    print('실행되지 않았거나 중단됐다 — 위 셀을 보라. 등록할 값이 없다.')
    if FAILURE:
        print(json.dumps(FAILURE, ensure_ascii=False, indent=2))
else:
    print(R.report_markdown(DECISION))
    print()
    print('--- 복사해 갈 값 ---')
    print('pinned commit      :', PINNED_COMMIT)
    print('NPZ SHA-256        :', DECISION['npz']['sha256'])
    print('corrective folder  :', DECISION['corrective_bundle']['directory'])
    print('corrective folderID:', DECISION['corrective_folder_id']['folder_id'])
    print('numpy 검증         :', DECISION['npz']['numpy_verification'])
    print()
    print(json.dumps(DECISION['verification']['observed'], indent=2,
                     sort_keys=True))

## 이 노트북이 하지 않는 것

`detect_r()` · beat join 재실행 · null 재실행 · M0~M4 집계 · DS2 per-beat label ·
V10 probability · association · S PR-AUC · 학습 · 기존 Drive 파일 이동·**삭제**·
덮어쓰기 · frozen 모듈 수정 · 12파일 계약 완화 · 값 등록.

**등록은 이 실행의 결과가 아니다.** corrective 폴더가 만들어져도 folder ID·
lineage·NPZ digest 는 별도 등록 PR 로만 들어가고, Q5-E PREP P1/P2 재실행은
그와 또 별개의 사용자 승인을 받는다.